# Phase 1 — Lesson 08: Optimization
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 08: Optimization
**Type:** Build | **Languages:** Python | **Time:** ~75 minutes
**Prerequisites:** Phase 1, Lessons 04-05 (Derivatives, Gradients)



# What You'll Learn

---

## Learning Objectives

- Implement vanilla gradient descent, SGD with momentum, and Adam from scratch
- Compare optimizer convergence on the Rosenbrock function and explain why Adam adapts per-weight learning rates
- Distinguish convex from non-convex loss landscapes and explain the role of saddle points in high dimensions
- Configure learning rate schedules (step decay, cosine annealing, warmup) for training stability


# The Problem

---

You have a loss function. It tells you how wrong your model is. You have gradients. They tell you which direction makes the loss worse. Now you need a strategy for walking downhill.

The naive approach is simple: move opposite the gradient. Scale the step by some number called the learning rate. Repeat. This is gradient descent, and it works.


# Gradient Descent

---

```
w = w - lr * gradient
```

That is the entire algorithm. One line. Compute the gradient, move in the opposite direction.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltdef rosenbrock(params):    x, y = params    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2def rosenbrock_grad(params):    x, y = params    df_dx = -2 * (1 - x) - 400 * x * (y - x ** 2)    df_dy = 200 * (y - x ** 2)    return np.array([df_dx, df_dy])start = np.array([-1.0, 1.0])lr = 0.0005params = start.copy()path = [params.copy()]for _ in range(2000):    params = params - lr * rosenbrock_grad(params)    path.append(params.copy())path = np.array(path)print(f"Final: x={params[0]:.4f}, y={params[1]:.4f}, loss={rosenbrock(params):.8f}")

In [ ]:
import matplotlib.pyplot as plt# Plot Rosenbrock landscape and GD pathx = np.linspace(-1.5, 1.5, 100)y = np.linspace(-0.5, 2.5, 100)X, Y = np.meshgrid(x, y)Z = (1 - X) ** 2 + 100 * (Y - X ** 2) ** 2fig, ax = plt.subplots(figsize=(7, 6))cont = ax.contour(X, Y, Z, levels=30, cmap='viridis', alpha=0.6)ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=3, label='GD path')ax.plot(path[0, 0], path[0, 1], 'ro', markersize=8, label='Start (-1, 1)')ax.plot(1, 1, 'g*', markersize=12, label='Minimum (1, 1)')ax.set_xlabel('x'); ax.set_ylabel('y')ax.set_title('Gradient Descent on Rosenbrock Function', fontsize=12, fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

# Learning Rate: The Most Important Hyperparameter

---

The learning rate controls step size:

| Too large | Too small | Just right |
|-----------|-----------|------------|
| Overshoots, diverges | Crawls, wastes compute | Converges efficiently |


# Momentum: The Ball Rolling Downhill

---

Vanilla gradient descent only looks at the current gradient. Momentum accumulates past gradients into a velocity:

```
v = momentum * v - lr * gradient
w = w + v
```

Typical momentum: 0.9. Dampens oscillations, accelerates through flat regions.


In [ ]:
import numpy as npdef sgd_momentum(params, grads, lr=0.0001, momentum=0.9, velocity=None):    if velocity is None:        velocity = np.zeros_like(params)    velocity = momentum * velocity - lr * grads    return params + velocity, velocitystart = np.array([-1.0, 1.0])params = start.copy()velocity = np.zeros(2)path_momentum = [params.copy()]for _ in range(2000):    grads = rosenbrock_grad(params)    params, velocity = sgd_momentum(params, grads, lr=0.0001, momentum=0.9, velocity=velocity)    path_momentum.append(params.copy())print(f"Momentum: x={params[0]:.4f}, y={params[1]:.4f}, loss={rosenbrock(params):.8f}")

# Adam: Adaptive Learning Rates

---

Adam tracks two things per weight:
- First moment (m): running average of gradients (like momentum)
- Second moment (v): running average of squared gradients

```
m = beta1 * m + (1 - beta1) * gradient
v = beta2 * v + (1 - beta2) * gradient²
w = w - lr * m_hat / (sqrt(v_hat) + epsilon)
```

Default: lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8.


In [ ]:
import numpy as npdef adam_step(params, grads, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, t=0, m=None, v=None):    if m is None: m = np.zeros_like(params)    if v is None: v = np.zeros_like(params)    t += 1    m = beta1 * m + (1 - beta1) * grads    v = beta2 * v + (1 - beta2) * grads ** 2    m_hat = m / (1 - beta1 ** t)    v_hat = v / (1 - beta2 ** t)    return params - lr * m_hat / (np.sqrt(v_hat) + eps), m, v, tstart = np.array([-1.0, 1.0])params = start.copy()m = v = Nonet = 0path_adam = [params.copy()]for _ in range(2000):    grads = rosenbrock_grad(params)    params, m, v, t = adam_step(params, grads, lr=0.01, m=m, v=v, t=t)    path_adam.append(params.copy())print(f"Adam: x={params[0]:.4f}, y={params[1]:.4f}, loss={rosenbrock(params):.8f}")

In [ ]:
import matplotlib.pyplot as plt# Compare all three optimizers on Rosenbrockfig, ax = plt.subplots(figsize=(7, 6))cont = ax.contour(X, Y, Z, levels=30, cmap='viridis', alpha=0.4)ax.plot(path[:, 0], path[:, 1], 'r.-', markersize=2, label='GD', alpha=0.7)ax.plot(path_momentum[:, 0], path_momentum[:, 1], 'g.-', markersize=2, label='Momentum', alpha=0.7)ax.plot(path_adam[:, 0], path_adam[:, 1], 'b.-', markersize=2, label='Adam', alpha=0.7)ax.plot(1, 1, 'k*', markersize=12, label='Minimum')ax.set_xlabel('x'); ax.set_ylabel('y')ax.set_title('Optimizer Comparison on Rosenbrock Function', fontsize=12, fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

# Convex vs Non-Convex

---

| Problem | Convex? | Why |
|---------|---------|-----|
| Linear regression (MSE) | Yes | Loss is quadratic in weights |
| Logistic regression | Yes | Log-loss is convex in weights |
| Neural network | No | Nonlinear activations create non-convex landscape |

In practice, local minima in high-dimensional neural networks are rarely a problem. Saddle points are the real obstacle.


# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What does the learning rate control?
> **Explanation:** Step size — too small = slow, too large = overshoot/diverge.

**Q2:** What does momentum add?
> **Explanation:** A persistent velocity that dampens oscillations and accelerates through flat regions.

**Q3:** What does Adam adapt?
> **Explanation:** Per-weight learning rates using running averages of gradients and squared gradients.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Run GD on Rosenbrock with learning rates [0.0001, 0.0005, 0.001, 0.005] and compare final loss.

**Exercise 2:** Compare momentum values [0.0, 0.5, 0.9, 0.99] on Rosenbrock convergence.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Gradient descent | Go downhill | Update weights by subtracting gradient × learning rate |
| Momentum | Keep rolling | Accumulate past gradients into a velocity vector |
| Adam | The default optimizer | Adaptive Moment Estimation — per-weight adaptive learning rates |
| Learning rate schedule | Change lr over time | Adjust learning rate during training; large steps early, small late |
| Convex function | One valley | Any local minimum is the global minimum |
| Saddle point | Flat but not a minimum | Gradient is zero but curvature is mixed |


# Further Reading

---

- [Sebastian Ruder: An overview of gradient descent optimization algorithms](https://ruder.io/optimizing-gradient-descent/)
- [Why Momentum Really Works (Distill)](https://distill.pub/2017/momentum/)


# Next Steps

---

When you're ready, continue to Lesson 09: Information Theory.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
